In [1]:
from src.dataloading.gnndataloader import GNNDataLoader
from src.utils import get_data_env

modelname       = 'gatv2model'
disease_name    = 'influenza'
model           = 'tgcn'
nuts_level      = 'nuts3'
min_date        ='2006-05-15'
max_date        = '2020-06-01'
split_trainval  = '2018-06-01'
split_valtest   = '2019-06-01'
split_berlin    = False


horizon_size    = 1
horizon_leadtime= 3
sequence_length = 1
lags            = 1


# training hparams
n_epochs        = 1
lr              = 0.0005
min_delta       = 0.0001
loss            = 'mse'

global_hparams = {
    "lr"                : lr,
    "n_epochs"          : n_epochs,
    "scheduler"         : 'plateau',
    "scheduler_kwargs"  :   {'mode': 'min', 'factor': 0.5, 'patience': 7},
    'min_delta'         : min_delta,
    'loss'              : loss,
    'patience'          : 20,
    }

epidata_loader_basis = GNNDataLoader(disease_name, get_data_env(), nuts_level=nuts_level, min_date=min_date,max_date=max_date, include_population=False, horizon_size = horizon_size, horizon_leadtime = horizon_leadtime, sequence_length=sequence_length, split_berlin=split_berlin)
epidata_loader_basis.add_time_features()
epidata_loader_basis.log_transform_target()
epidata_loader_basis.set_splits(split_trainval, split_valtest)
epidata_loader_basis.normalize()
epidata_loader_basis.add_lagged_features(lags = lags)
epidata_loader_basis.finalize()

Dataloader temporal windowing: extending data collection from 2006-05-15 to 2006-04-17 (+4 weeks)
berlin districts removed


EpiDataLoader(disease=influenza, nuts_level=nuts3, min_date=2006-04-17, max_date=2020-06-01, horizon_size=1, horizon_leadtime=3, sequence_length=1)

In [2]:
from src.new_models_module.deep.gatv2 import GATv2Model
from src.new_models_module.deep.tgcn import TGCNModel

/home/de-schrijvers/.conda/envs/gnenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dl = epidata_loader_basis.copy().retrieve_graph('identity_selfmax').construct_dataloaders()

ml = GATv2Model(dataloader= dl)
ml.set_model_hparams()
ml.set_global_hparams(**global_hparams)
# ml.train(2)
# ml.forecast('test')
# ml.show_forecasts('test', 25)

Dataloader temporal windowing: extending data collection from 2006-05-15 to 2006-04-17 (+4 weeks)
berlin districts removed


In [4]:
print(ml)

<DeepModel(
    model name         : unknown
    model class        : GATv2Model

    ----------- STATUS --------------
    model_initialized  : ✓
    global_hparams_set : ✓
    trained            : ✗
    forecasted         : ✗

    ----------- FORECASTS -----------
    forecasted         : []

    ----------- MODEL HPARAMS -------
    hidden_size        : 64
    num_layers         : 2
    temporal_layers    : 2
    dropout            : 0.2
    strategy           : recurrent strategy

    ----------- GLOBAL HPARAMS ------
    lr                 : 0.0005
    n_epochs           : 1
    patience           : 20
    min_delta          : 0.0001
    optimizer          : adam
    optimizer_kwargs   : None
    scheduler          : plateau
    scheduler_kwargs   : {'mode': 'min', 'factor': 0.5, 'patience': 7}
    loss               : mse
    loss_kwargs        : None
)>


In [5]:
dl = epidata_loader_basis.copy().retrieve_graph('identity_selfmax').construct_dataloaders()

ml = TGCNModel(dataloader= dl)
ml.set_model_hparams()
ml.set_global_hparams()
# ml.train(2)
# ml.forecast('test')
# ml.show_forecasts('test', 25)

Dataloader temporal windowing: extending data collection from 2006-05-15 to 2006-04-17 (+4 weeks)
berlin districts removed


In [6]:
ml.strategy

standard strategy